# Demo: Building a RAG-powered FAQ Agent with Custom Knowledge

# Step 1: Install required packages

In [ ]:
!pip install -U langchain langchain-openai langchain-community langchain-classic faiss-cpu tiktoken

# Step 2: Import dependencies

In [ ]:
import os
from langchain_openai import AzureChatOpenAI, AzureOpenAIEmbeddings
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import CharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_classic.chains import RetrievalQA
#update

# Step 3: Set Azure OpenAI credentials



In [ ]:
os.environ["AZURE_OPENAI_API_KEY"] = "2ABecnfxzhRg4M5D6pBKiqxXVhmGB2WvQ0aYKkbTCPsj0JLKsZPfJQQJ99BDAC77bzfXJ3w3AAABACOGi3sC"

# Step 4: Load and chunk your custom FAQ document


In [ ]:
# Load the FAQ document and split it into chunks for embedding

loader = TextLoader("faq.txt")  # Ensure this file exists
documents = loader.load()

text_splitter = CharacterTextSplitter(chunk_size=500, chunk_overlap=50)
docs = text_splitter.split_documents(documents)

# Step 5: Create vectorstore using Azure embeddings


In [ ]:
# Embed document chunks and store them in a FAISS vector index

embeddings = AzureOpenAIEmbeddings(
    azure_endpoint="https://openai-api-management-gw.azure-api.net",
    api_version="2023-05-15",
    deployment="text-embedding-ada-002",
    api_key=os.environ["AZURE_OPENAI_API_KEY"]
)

vectorstore = FAISS.from_documents(docs, embeddings)

# Step 6: Initialize the Azure OpenAI LLM

In [ ]:
# Initialize the GPT-4o model from Azure with temperature 0 for deterministic output

llm = AzureChatOpenAI(
    azure_endpoint="https://openai-api-management-gw.azure-api.net",
    api_version="2025-01-01-preview",
    deployment_name="gpt-5-mini"
)
#update

# Step 7: Create the RAG chain

In [ ]:
# Create a RetrievalQA chain that uses the retriever and LLM to answer queries with source context

qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=vectorstore.as_retriever(),
    return_source_documents=True
)

# Step 8: Ask a question

In [ ]:
# Send a question to the RAG chain and store the result

query = "What is our return policy?"
result = qa_chain.invoke({"query": query})

# Step 9: Print results

In [ ]:
# Display the final answer and the source chunks used

print("Answer:", result["result"])
print("\n--- Sources ---")
for i, doc in enumerate(result["source_documents"], 1):
    print(f"\nSource {i}:")
    print(doc.page_content)